# ⚡ Pipeline: Compuertas Lógicas desde Cero
**End-to-end: Logic Gates from Scratch**  
Diplomado Superior en Redes Neuronales Artificiales y Deep Learning

---
> Desde las tablas de verdad hasta circuitos sumadores completos.  
> Demostramos que NAND es universal y construimos un sumador de 2 bits.

---
## 📐 De la Tabla de Verdad al Circuito

Toda compuerta lógica se define por su **tabla de verdad**: la salida para cada  
combinación posible de entradas. A partir de ahí, construimos circuitos combinacionales.

| Compuerta | Fórmula | Símbolo |
|-----------|---------|--------|
| AND | A · B | `A & B` |
| OR | A + B | `A | B` |
| NOT | ¬A | `~A` |
| NAND | ¬(A · B) | `~(A & B)` |
| NOR | ¬(A + B) | `~(A | B)` |
| XOR | A ⊕ B | `A ^ B` |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..')))
from libreria_modulo5 import *

---
## 🧮 Tablas de Verdad: AND, OR, NOT

In [ ]:
tabla_verdad(compuerta_and, 'AND')
tabla_verdad(compuerta_or, 'OR')
tabla_verdad(compuerta_not, 'NOT', n_entradas=1)

---
## 🔄 NAND es Universal

Demostramos que cualquier compuerta se puede construir solo con compuertas NAND.

In [ ]:
# NOT desde NAND
compuerta_not_con_nand = lambda a: compuerta_nand(a, a)

print("NOT construido con NAND:")
for a in [0, 1]:
    print(f"  NAND({a}, {a}) = {compuerta_not_con_nand(a)} (NOT = {compuerta_not(a)})")

In [ ]:
# AND desde NAND
compuerta_and_con_nand = lambda a, b: compuerta_nand(compuerta_nand(a, b), compuerta_nand(a, b))

print("AND construido con NAND:")
for a in [0, 1]:
    for b in [0, 1]:
        r = compuerta_and_con_nand(a, b)
        esperado = compuerta_and(a, b)
        print(f"  A={a} B={b} → {r} (esperado={esperado}) {'✅' if r == esperado else '❌'}")

In [ ]:
# OR desde NAND
compuerta_or_con_nand = lambda a, b: compuerta_nand(compuerta_nand(a, a), compuerta_nand(b, b))

print("OR construido con NAND:")
for a in [0, 1]:
    for b in [0, 1]:
        r = compuerta_or_con_nand(a, b)
        esperado = compuerta_or(a, b)
        print(f"  A={a} B={b} → {r} (esperado={esperado}) {'✅' if r == esperado else '❌'}")

---
## ➕ Circuito Sumador Medio (Half Adder)

Suma 2 bits individuales. Produce **suma** (XOR) y **acarreo** (AND).

In [ ]:
print("=== Sumador Medio (Half Adder) ===")
print("  A   B | Suma  Acarreo")
print("  ------+-------------")
for a in [0, 1]:
    for b in [0, 1]:
        r = circuito_sumador_medio(a, b)
        print(f"  {a}   {b} |  {r['suma']}     {r['acarreo']}")

---
## ➕➕ Sumador Completo (Full Adder)

Suma 3 bits: A + B + acarreo de entrada. Es el bloque base de los sumadores de n bits.

In [ ]:
print("=== Sumador Completo (Full Adder) ===")
print("  A   B  Cin | Suma  Cout")
print("  ----------+------------")
for a in [0, 1]:
    for b in [0, 1]:
        for cin in [0, 1]:
            r = circuito_sumador_completo(a, b, cin)
            print(f"  {a}   {b}   {cin} |  {r['suma']}     {r['acarreo']}")

---
## 🔢 Sumador de 2 bits (Cascada de Full Adders)

Conectamos dos sumadores completos en cascada para sumar números de 2 bits:  
`A1A0 + B1B0 = S1S0` con acarreo final.

In [ ]:
def sumador_2bits(a1, a0, b1, b0):
    """Suma dos números de 2 bits usando full adders en cascada."""
    fa0 = circuito_sumador_completo(a0, b0, 0)
    fa1 = circuito_sumador_completo(a1, b1, fa0['acarreo'])
    return {
        'bits': [fa1['suma'], fa0['suma']],
        'acarreo': fa1['acarreo'],
        'decimal': binario_a_decimal([fa1['suma'], fa0['suma']]),
        'acarreo_final': fa1['acarreo']
    }

print("=== Sumador de 2 bits ===")
print("  A1 A0  B1 B0 | S1 S0  Carry | Decimal")
print("  -------------+---------------+--------")
for a in range(4):
    for b in range(4):
        a1, a0 = decimal_a_binario(a, 2)
        b1, b0 = decimal_a_binario(b, 2)
        r = sumador_2bits(a1, a0, b1, b0)
        bits = r['bits']
        print(f"  {a1}  {a0}   {b1}  {b0} |  {bits[0]}  {bits[1]}   {r['acarreo_final']}     | {a} + {b} = {r['decimal']} (esperado {a+b}) {'✅' if r['decimal'] == a+b else '❌'}")

---
## 📋 Generador Universal de Tablas de Verdad

Probamos cualquier circuito de forma genérica.

In [ ]:
def generar_tabla_verdad(circuito, nombre, n_entradas=2, nom_entradas=['A', 'B']):
    """Genera tabla de verdad para cualquier circuito."""
    print(f"\n{'='*40}")
    print(f"  Tabla de verdad: {nombre}")
    print(f"{'='*40}")
    
    total = 2 ** n_entradas
    for i in range(total):
        bits = decimal_a_binario(i, n_entradas)
        salida = circuito(*bits)
        entradas_str = ' '.join(str(b) for b in bits)
        print(f"  {entradas_str} → {salida}")

print("Circuito XOR construido desde AND/OR/NOT:")
xor_personalizado = construir_xor_desde_and_or_not()
generar_tabla_verdad(xor_personalizado, "XOR (con AND/OR/NOT)")

In [ ]:
# Verificar que XOR coincide con la función nativa
print("Verificación XOR nativa vs construida:")
xor_personalizado = construir_xor_desde_and_or_not()
for a in [0, 1]:
    for b in [0, 1]:
        r1 = compuerta_xor(a, b)
        r2 = xor_personalizado(a, b)
        ok = '✅' if r1 == r2 else '❌'
        print(f"  XOR({a},{b}) = nativa:{r1} construida:{r2} {ok}")

---
## 🔄 Verificación de De Morgan

Leyes de De Morgan:
- `¬(A · B) = ¬A + ¬B`  (NAND = NOT OR)
- `¬(A + B) = ¬A · ¬B`  (NOR = AND NOT)

In [ ]:
print("=== Verificación: 1ra Ley de De Morgan ===")
print("  ¬(A · B) == ¬A + ¬B")
print()
for a in [0, 1]:
    for b in [0, 1]:
        lhs = compuerta_nand(a, b)
        rhs = compuerta_or(compuerta_not(a), compuerta_not(b))
        ok = '✅' if lhs == rhs else '❌'
        print(f"  A={a} B={b}: NAND={lhs}  NOT-A+NOT-B={rhs}  {ok}")

In [ ]:
print("=== Verificación: 2da Ley de De Morgan ===")
print("  ¬(A + B) == ¬A · ¬B")
print()
for a in [0, 1]:
    for b in [0, 1]:
        lhs = compuerta_nor(a, b)
        rhs = compuerta_and(compuerta_not(a), compuerta_not(b))
        ok = '✅' if lhs == rhs else '❌'
        print(f"  A={a} B={b}: NOR={lhs}  NOT-A·NOT-B={rhs}  {ok}")

---
## 🧪 #TODO: Ejercicios

1. **Sumador de 4 bits**: Extiende `sumador_2bits` a 4 bits en cascada.  
   Verifica que `sumador_4bits(bin(10), bin(5)) == 15`.

2. **Medio restador**: Diseña un `circuito_restador_medio` usando XOR y NOT-AND.  
   ¿Cuál es la tabla de verdad?

3. **Multiplexor 2:1**: Construye un MUX con compuertas lógicas:  
   `S = (A · ¬sel) + (B · sel)`

4. **Comparador de 1 bit**: Diseña un circuito que indique si A > B usando solo AND/OR/NOT.

5. **Circuito majority**: Un circuito que devuelve 1 si la mayoría de las entradas son 1.  
   Pruébalo para 3 entradas.